# image-object-detection: image_object_detection (batch, custom-pipeline)

**Use case:** `usecase-type/batch/computer-vision/object-detection/azureml-pipeline` — bounded scoring of many requests into stored predictions with the Azure ML pipeline with custom training.

**Training:** Factory Azure ML v2 pipeline with a custom torchvision training step and the vision quality gate.

**Serving:** Azure ML batch endpoint (no-code MLflow) or factory scoring pipeline job via `serving-render`.

**Status:** Executable notebook templates over shared engines; cloud steps are explicit, charged operations behind switches that default to false and need the listed prerequisites.

Kaggle-only source: `andrewmvd/car-plate-detection`.
Dataset status: **configured**.
433 images with Pascal VOC bounding boxes. Detection is not OCR or identity recognition. Review privacy and regional representativeness before any production use.

**Prerequisites**

- Review the scenario's dataset status, license and notes; gated datasets stay blocked until approved.
- Install the factory extras `train` (plus `vision` for images, `azure` for Azure ML steps and `streaming` for Event Hubs consumers).
- For Azure steps: `user-config/runtime.local.json` naming an existing workspace, compute, identity and storage selection.
- Image training in Azure needs `gpu_compute`, or `vision.device: cpu` for the small custom example.

**Limitations**

- Templates are not evidence of a completed Kaggle, Azure or Databricks run.

This notebook is an executable template, not evidence of a completed Kaggle or Azure run.
Install the factory's declared extras into this kernel separately. No packages,
credentials, cloud resources, or competition terms are installed/accepted by this notebook.
All execution switches default to false. Run **either** SDK v2 **or** CLI v2 submission, not both.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "pyproject.toml").is_file()
             and (p / "accelerator" / "src" / "ml_model_factory" / "config.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the 50-ml-model-factory checkout")
SOURCE = ROOT / "accelerator" / "src"
sys.path.insert(0, str(SOURCE))
from ml_model_factory.config import load_json, validate_scenario, validate_runtime, write_json

SCENARIO = ROOT / "user-config" / "model" / "scenarios" / "image-object-detection.json"
scenario = validate_scenario(load_json(SCENARIO))
MODE = "custom"
PATTERN = "batch"
TECHNOLOGY = "azureml-pipeline"
RUNTIME = Path("user-config/runtime.local.json")
if not RUNTIME.is_absolute():
    RUNTIME = ROOT / RUNTIME
WORK = ROOT / "ml-environment" / "outputs" / scenario["name"] / "custom-pipeline"
RAW = ROOT / "data" / "in" / scenario["name"]
PREPARED = ROOT / "data" / "out" / scenario["name"] / "custom-pipeline" / "prepared"
MODEL = WORK / "model"
EVALUATION = WORK / "evaluation"
RENDERED = WORK / "azureml"
INFERENCE = WORK / PATTERN
REQUESTS = INFERENCE / "requests"
MODEL_NAME = scenario.get("model_name", scenario["name"])
MODEL_ID = ""  # immutable azureml:<name>:<version> after quality-gated registration
RUN_INGEST = False
RUN_PREPARE = False
USE_LAKE = False
RUN_LAKE_TRAIN = False
RENDER_AZURE = False
SUBMIT_SDK = False
SUBMIT_CLI = False
REGISTER_MODEL = False
SAMPLE_REQUESTS = False
RENDER_BATCH = False
DEPLOY_BATCH = False
INVOKE_BATCH = False
PIPELINE_JOB_NAME = ""
BATCH_ENDPOINT = None
BATCH_INPUT_URI = ""  # credential-free URI of uploaded requests
LAKE_CONFIG = ROOT / "user-config" / "lake.local.json"
LAKE_ROOT = ROOT / "data" / "out" / "lake-data"
EFFECTIVE_RUNTIME = RUNTIME
FACTORY_ENV = os.environ.copy()
FACTORY_ENV["PYTHONPATH"] = str(SOURCE) + os.pathsep + FACTORY_ENV.get("PYTHONPATH", "")
FACTORY_ENV.setdefault("MLFLOW_TRACKING_URI", (ROOT / "ml-environment" / "mlruns").as_uri())

def factory(*arguments):
    command = [sys.executable, "-m", "ml_model_factory", *map(str, arguments)]
    subprocess.run(command, cwd=ROOT, env=FACTORY_ENV, check=True)


## Kaggle ingestion and reproducible preparation

Review the source license and any competition rules first. Required-selection and
license-review gates block ingestion. Do not clear them without documented approval.
The download manifest records file hashes and pinned dataset version. Existing nonempty
outputs are never silently replaced. Titanic preparation additionally writes `titanic.parquet`.
Forecasting uses chronological holdouts, not random rows.


In [ ]:
if RUN_INGEST:
    factory("ingest", "--scenario", SCENARIO, "--output", RAW)
INPUT = RAW / scenario["dataset"]["file"]
if RUN_PREPARE:
    factory("prepare", "--scenario", SCENARIO, "--input", INPUT, "--output", PREPARED)
if (PREPARED / "manifest.json").exists():
    print(json.dumps(load_json(PREPARED / "manifest.json"), indent=2))


## Image portability and AutoML boundaries

Custom manifests reference copied split-local images and remain valid after relocation.
For AutoML, set `vision.image_base_uri` to an **immutable remote prepared-directory root**,
prepare again, and upload the entire prepared tree there without changing the layout.
Remote URLs must use identity-based access, not embedded SAS/credentials. This notebook
does not silently upload image data or replace URIs with job-local mount paths.

With `runtime.lake` (with a datastore) and a pinned `automl_vision_environment`, AutoML image
rendering binds the prepared images to the lake and produces a quality-gated pipeline (prepare,
AutoML, evaluate) whose evaluator uses the documented AutoML image scoring schema. Without both,
only a standalone AutoML training job is rendered, which cannot be registered.


## Versioned project/use-case lake (optional)

Set the project/environment, dataset version, snapshot ID and unique run ID in
`user-config/lake.local.json`. `lake-plan` is read-only. `lake-train` publishes source, gold snapshots,
models and evaluations locally with completion manifests; it does not upload data.
Keep unlabeled inference requests and observed feedback separate. Use `lake-infer` with
an immutable model version and unique request IDs, and `lake-feedback` for reviewed labels.
Streaming checkpoints are independent of execution run IDs and model versions.

Azure preparation uses a per-run working area and records the snapshot reference;
it does not claim that binding an output URI alone publishes an immutable snapshot.


In [ ]:
if USE_LAKE:
    factory("lake-plan", "--scenario", SCENARIO, "--config", LAKE_CONFIG)
    if RENDER_AZURE:
        runtime_with_lake = validate_runtime(load_json(RUNTIME))
        runtime_with_lake["lake"] = load_json(LAKE_CONFIG)
        EFFECTIVE_RUNTIME = WORK / "runtime.local.json"
        write_json(EFFECTIVE_RUNTIME, runtime_with_lake)
if RUN_LAKE_TRAIN:
    if not USE_LAKE or MODE != "custom":
        raise ValueError("Local lake training requires USE_LAKE and custom mode")
    factory("lake-train", "--scenario", SCENARIO, "--config", LAKE_CONFIG,
            "--root", LAKE_ROOT, "--input", INPUT)


## Render portable Azure ML v2 jobs (offline)

`user-config/runtime.local.json` must name existing subscription, workspace and compute resources and a
resolvable input data path. Resource discovery is read-only. Rendering creates files,
not Azure resources. Submission incurs compute charges. Image AutoML additionally
requires a GPU-compatible compute target and the prepared remote image layout above.


In [ ]:
if RENDER_AZURE:
    validate_runtime(load_json(EFFECTIVE_RUNTIME))
    factory("render", "--scenario", SCENARIO, "--runtime", EFFECTIVE_RUNTIME,
            "--mode", MODE, "--output", RENDERED)
JOB_PATH = RENDERED / "pipeline.yml" if (RENDERED / "pipeline.yml").exists() else RENDERED / "job.yml"


## Azure ML SDK v2 submission — alternative A


In [ ]:
if SUBMIT_SDK:
    if SUBMIT_CLI:
        raise ValueError("Choose one submission route to avoid duplicate charged jobs")
    from ml_model_factory.azureml import submit
    runtime = validate_runtime(load_json(EFFECTIVE_RUNTIME))
    if not JOB_PATH.exists():
        raise RuntimeError("Render and review the job before submission")
    # Shared SDK v2 submission binds the configured tenant and rejects unresolved inputs.
    print(submit(JOB_PATH, runtime))


## Azure ML CLI v2 submission — alternative B


In [ ]:
if SUBMIT_CLI:
    if SUBMIT_SDK:
        raise ValueError("Choose one submission route to avoid duplicate charged jobs")
    runtime = validate_runtime(load_json(EFFECTIVE_RUNTIME))
    if not JOB_PATH.exists():
        raise RuntimeError("Render and review the job before submission")
    subprocess.run([sys.executable, str(ROOT / "accelerator" / "scripts" / "azureml_cli.py"),
                    "--runtime", str(EFFECTIVE_RUNTIME), "--job", str(JOB_PATH)],
                   cwd=ROOT, env=FACTORY_ENV, check=True)

# Equivalent factory SDK-backed submission:
# factory("submit", "--runtime", RUNTIME, "--job", JOB_PATH)


## Quality-gated model registration

Only a completed factory pipeline whose `evaluate` step wrote a passing `quality-gate.json`
can register its named model output. Standalone training jobs are not a quality gate.
Set `PIPELINE_JOB_NAME` to the completed job, then copy the printed immutable model ID
into `MODEL_ID`. Registration is not production promotion.


In [ ]:
if REGISTER_MODEL:
    if not PIPELINE_JOB_NAME:
        raise ValueError("Set PIPELINE_JOB_NAME to the completed, evaluated factory pipeline job")
    subprocess.run([sys.executable, str(ROOT / "accelerator" / "scripts" / "azureml_sdk.py"),
                    "--runtime", str(EFFECTIVE_RUNTIME), "register", "--job-name", PIPELINE_JOB_NAME,
                    "--model-name", MODEL_NAME], cwd=ROOT, env=FACTORY_ENV, check=True)
print("MODEL_ID:", MODEL_ID or "not set")


## Unlabeled requests from the held-out split

`sample-requests` writes demonstration requests without labels; `requests.parquet` is the unlabeled batch input.
`labels.parquet` keeps held-out truth separately for monitoring demos, and forecasting
also writes `history.parquet` for AutoML scoring. Never send labels to a model.


In [ ]:
if SAMPLE_REQUESTS:
    factory("sample-requests", "--scenario", SCENARIO, "--prepared", PREPARED, "--output", REQUESTS, "--rows", 10)
if (REQUESTS / "manifest.json").exists():
    print(json.dumps(load_json(REQUESTS / "manifest.json"), indent=2))


## Azure ML batch serving

`serving-render --kind batch` renders a no-code MLflow batch endpoint for tabular
classification/regression and a factory scoring pipeline job otherwise (forecasting,
images, AutoML forecast with history). Upload the unlabeled requests to the selected
storage first and set `BATCH_INPUT_URI` to that credential-free URI (a folder input must
contain `data.parquet`). Scoring-job bundles have no persistent deployment: each
invocation resolves the registered model and inputs into a new job file and submits it.


In [ ]:
BATCH_BUNDLE = INFERENCE / "azureml-batch"
if RENDER_BATCH:
    factory("serving-render", "--scenario", SCENARIO, "--runtime", EFFECTIVE_RUNTIME, "--kind", "batch",
            "--model-name", MODEL_NAME, "--mode", MODE, "--output", BATCH_BUNDLE)
if BATCH_ENDPOINT is None and (BATCH_BUNDLE / "manifest.json").exists():
    BATCH_ENDPOINT = load_json(BATCH_BUNDLE / "manifest.json").get("endpoint_name")
if DEPLOY_BATCH:
    if not MODEL_ID:
        raise ValueError("Set MODEL_ID to the immutable registered model version")
    if BATCH_ENDPOINT:
        factory("serving-deploy", "--runtime", EFFECTIVE_RUNTIME, "--bundle", BATCH_BUNDLE,
                "--model-id", MODEL_ID, "--execute")
    else:
        print("Scoring-job bundle: no persistent deployment; INVOKE_BATCH submits a resolved job")
if INVOKE_BATCH:
    if not BATCH_INPUT_URI:
        raise ValueError("Upload unlabeled requests and set BATCH_INPUT_URI")
    if BATCH_ENDPOINT:
        factory("serving-invoke", "--runtime", EFFECTIVE_RUNTIME, "--kind", "batch", "--endpoint", BATCH_ENDPOINT,
                "--input", BATCH_INPUT_URI, "--execute")
    else:
        factory("serving-invoke", "--runtime", EFFECTIVE_RUNTIME, "--kind", "batch", "--bundle", BATCH_BUNDLE,
                "--model-id", MODEL_ID, "--input", BATCH_INPUT_URI, "--execute")


## Monitoring and clean-up

Join predictions with observed labels by `request_id` only after outcomes are known, then use
`python -m ml_model_factory monitor` with explicit reference/current windows (see the
model-factory guide). Predictions are never labels. Batch compute scales back to zero; storage of inputs and predictions remains.
